Défi quotidien : Gestion et analyse des données en Python


In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# Étape 1 : Importation du jeu de données
# ==========================================
# Remplacez 'ds_salaries.csv' par le chemin exact de votre fichier téléchargé
try:
    df = pd.read_csv("ds_salaries.csv")
    print("✅ Jeu de données importé avec succès.")
except FileNotFoundError:
    print(
        "❌ Fichier introuvable. Génération d'un jeu de données fictif pour la démonstration..."
    )
    # Simulation de données conformes à la structure classique du dataset Kaggle
    np.random.seed(42)
    data = {
        "experience_level": np.random.choice(
            ["EN", "MI", "SE", "EX"], size=200, p=[0.2, 0.3, 0.4, 0.1]
        ),
        "salary_in_usd": np.random.randint(40000, 250000, size=200),
        "remote_ratio": np.random.choice([0, 50, 100], size=200),
        "company_size": np.random.choice(["S", "M", "L"], size=200),
        "work_year": np.random.choice([2022, 2023, 2024], size=200),
    }
    df = pd.DataFrame(data)

# Visualisation des premières lignes
print(df.head(), "\n" + "=" * 50)


# ==========================================
# Étape 2 : Normalisation Min-Max du salaire
# ==========================================
# Initialisation du scaler pour transformer les valeurs entre 0 et 1
scaler = MinMaxScaler()

# Application sur la colonne des salaires en USD (référence standard du dataset)
# .values.reshape(-1, 1) est requis car Scikit-Learn attend un tableau 2D
df["salary_normalized"] = scaler.fit_transform(
    df[["salary_in_usd"]].values.reshape(-1, 1)
)

print("✅ Étape 2 : Normalisation Min-Max réussie.")
print(df[["salary_in_usd", "salary_normalized"]].head(), "\n" + "=" * 50)


# ==========================================
# Étape 3 : Réduction de dimensionnalité (PCA)
# ==========================================
# 1. Sélection et encodage des colonnes numériques ou catégorielles pour la PCA
# La PCA nécessite uniquement des valeurs numériques
features_to_encode = ["experience_level", "company_size"]
df_encoded = pd.get_dummies(
    df.drop(columns=["salary_in_usd"]),
    columns=features_to_encode,
    drop_first=True,
)

# Remplacement des valeurs booléennes générées par get_dummies en 0 et 1
df_encoded = df_encoded.astype(float)

# 2. Application de la PCA pour réduire le jeu de données à 2 composantes principales
pca = PCA(n_components=2)
pca_features = pca.fit_transform(df_encoded)

# Ajout des composantes au DataFrame principal
df["PCA_1"] = pca_features[:, 0]
df["PCA_2"] = pca_features[:, 1]

print("✅ Étape 3 : Réduction de dimensionnalité (PCA) exécutée.")
print(
    f"Variance expliquée par les 2 composantes : {pca.explained_variance_ratio_}"
)
print(df[["PCA_1", "PCA_2"]].head(), "\n" + "=" * 50)


# ==========================================
# Étape 4 : Agrégation par niveau d'expérience
# ==========================================
# Groupement et calcul simultané de la moyenne et de la médiane
salary_stats = (
    df.groupby("experience_level")["salary_in_usd"]
    .agg(["mean", "median"])
    .reset_index()
)

# Renommer les colonnes pour plus de clarté
salary_stats.columns = [
    "Niveau d'expérience",
    "Salaire Moyen ($)",
    "Salaire Médian ($)",
]

print("✅ Étape 4 : Agrégation et analyse des salaires par expérience :")
print(salary_stats.to_string(index=False))


❌ Fichier introuvable. Génération d'un jeu de données fictif pour la démonstration...
  experience_level  salary_in_usd  remote_ratio company_size  work_year
0               MI          52183             0            M       2024
1               EX         200371           100            M       2024
2               SE         183946           100            M       2024
3               SE         156336             0            S       2022
4               EN          72711            50            L       2024 
✅ Étape 2 : Normalisation Min-Max réussie.
   salary_in_usd  salary_normalized
0          52183           0.047333
1         200371           0.761971
2         183946           0.682761
3         156336           0.549612
4          72711           0.146329 
✅ Étape 3 : Réduction de dimensionnalité (PCA) exécutée.
Variance expliquée par les 2 composantes : [9.99007583e-01 3.78730581e-04]
       PCA_1     PCA_2
0 -45.498571  0.894821
1  54.500441  0.928867
2  54.501406  0.9509